# Adding Labor Market Information (LMI) to the CareerNet Dataset

This notebook guides you through the process of obtaining [Labor Market Information (LMI) from the Bureau of Labor Statistics (BLS)](https://www.bls.gov/oes/tables.htm) and integrating it into the [CareerNet dataset](https://github.com/RenaissancePhilanthropy/careernet-data).

By completing this process, you will enrich the CareerNet data with real-world employment figures and salary expectations by joining both datasets using their Standard Occupational Classification (SOC) codes.

This notebook uses **Metropolitan Statistical Area (MSA)** and **Balance of State (BOS)** files to provide employment and wage data at the metro and non-metro area level. This allows you to filter by both state and a specific geographic area (e.g., St. Louis, MO-IL).

*Note: This notebook focuses on total employment, location quotient, jobs per 1,000 workers, annual mean wage, annual median wage, and the 25th and 75th percentile wages. The code can be adjusted to incorporate other BLS LMI data of interest.*

In [ ]:
# Install required packages if they are not already installed in this environment
%pip install -q pandas numpy openpyxl

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import warnings

# Suppress openpyxl warnings for default styles
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

## Step 1: Obtain the LMI Dataset from the BLS

The Bureau of Labor Statistics (BLS) provides Occupational Employment and Wage Statistics (OES) at the metro and non-metro area level in two files:

- **MSA file** (`MSA_M####_dl.xlsx`): Metropolitan Statistical Areas — major metro areas such as *St. Louis, MO-IL* or *Kansas City, MO-KS*.
- **BOS file** (`BOS_M####_dl.xlsx`): Balance of State — non-metropolitan (rural) areas not captured in MSA data.

Because the BLS frequently blocks automated Python downloads (HTTP 403 errors), it is best to download these files manually.

1. Navigate to the BLS OES Tables page: [https://www.bls.gov/oes/tables.htm](https://www.bls.gov/oes/tables.htm)
2. Look for the most recent **Metropolitan area** data — download both the MSA (`MSA_M####_dl.xlsx`) and BOS (`BOS_M####_dl.xlsx`) Excel files.
3. Place both downloaded `.xlsx` files in the **same directory** as this Jupyter Notebook.

*Running in Google Colab?* "The same directory as this notebook" means Colab's default `/content` folder. Open the **Files** panel (folder icon in the left sidebar) and upload the file(s) there. Uploaded files are deleted when the runtime resets, so you will need to upload them again in each new session.

*Note: Update `BLS_YEAR_SUFFIX` in the configuration cell (Step 2) to match your downloaded files.*

## Step 1b: Discover Available Areas by State

Before configuring your filters, run this cell to see every MSA and BOS area available for a given state. The output shows the **exact area name** you must copy into the `TARGET_AREA` field in Step 2.

In [ ]:
# ==============================================================================
# DISCOVERY: Explore available areas for a given state
# ==============================================================================
DISCOVERY_STATE = 'MO'            # Two-letter state code to explore
BLS_YEAR_SUFFIX = 'M2025_dl.xlsx' # Must match Step 2 BLS_YEAR_SUFFIX exactly
# ==============================================================================

bos_file = f'BOS_{BLS_YEAR_SUFFIX}'
msa_file = f'MSA_{BLS_YEAR_SUFFIX}'

try:
    bos_disc = pd.read_excel(bos_file)
    msa_disc = pd.read_excel(msa_file)

    msa_areas = (
        msa_disc[msa_disc['PRIM_STATE'] == DISCOVERY_STATE][['AREA_TITLE']]
        .drop_duplicates()
        .sort_values('AREA_TITLE')
    )
    bos_areas = (
        bos_disc[bos_disc['PRIM_STATE'] == DISCOVERY_STATE][['AREA_TITLE']]
        .drop_duplicates()
        .sort_values('AREA_TITLE')
    )

    if msa_areas.empty and bos_areas.empty:
        print(f"No areas found for state '{DISCOVERY_STATE}'. Verify the two-letter state code.")
    else:
        if not msa_areas.empty:
            print(f"Metropolitan Statistical Areas (MSA) in {DISCOVERY_STATE}:")
            for area in msa_areas['AREA_TITLE']:
                print(f"  {area}")
        if not bos_areas.empty:
            print(f"\nBalance of State / Non-metro Areas (BOS) in {DISCOVERY_STATE}:")
            for area in bos_areas['AREA_TITLE']:
                print(f"  {area}")
        print(f"\nCopy one of the names above and set it as TARGET_AREA in Step 2.")

except FileNotFoundError as e:
    print(f"File not found: {e}")
    print(f"Ensure both {bos_file} and {msa_file} are in the same directory as this notebook.")
except Exception as e:
    print(f"Error: {e}")

## Step 2: Load and Filter the BLS Area Data

Configure the state and area you want to analyze. Both the MSA and BOS files are loaded and combined, then filtered to your selections. The data is also automatically filtered to cross-industry aggregate rows, which represent totals across all industries for each occupation. This leaves one LMI record per SOC code per area. With a `TARGET_AREA` set, each CareerNet row matches at most one BLS record, so the merge does not add rows. With `TARGET_AREA = ''`, each matched row is repeated once per area in the state. For example, the 12 MO areas turn 32,964 exploded `general` rows into 152,017. The MSA and BOS files include only major and detailed SOC codes, so CareerNet's minor and broad codes (e.g. `29-1000`, `29-1210`) receive no LMI data.

Leave `TARGET_AREA` as an empty string (`''`) to keep **all areas** within the target state.

*Note: Update the variables below to match your data of interest.*
- `TARGET_STATE` — two-letter state code (e.g., `'MO'`)
- `TARGET_AREA`  — exact area title from the discovery cell above (e.g., `'St. Louis, MO-IL'`), or `''` for all areas in the state
- `BLS_YEAR_SUFFIX` — year/name suffix of your BLS files (e.g., `'M2025_dl.xlsx'`)

In [ ]:
# ==============================================================================
# CONFIGURATION: BLS Area Data Settings
# ==============================================================================
# 1. Two-letter state code
TARGET_STATE = 'MO'

# 2. Exact area title from the discovery cell (Step 1b).
#    Set to '' to include ALL areas in the target state.
TARGET_AREA = 'St. Louis, MO-IL'

# 3. (Optional) Update this suffix if you download a different year from the BLS.
#    Also update BLS_YEAR_SUFFIX in the discovery cell (Step 1b) to match.
BLS_YEAR_SUFFIX = 'M2025_dl.xlsx'
# ==============================================================================

bos_file = f'BOS_{BLS_YEAR_SUFFIX}'
msa_file = f'MSA_{BLS_YEAR_SUFFIX}'

print(f"Loading {bos_file} and {msa_file}...")

try:
    bos_df = pd.read_excel(bos_file)
    msa_df = pd.read_excel(msa_file)
    combined_df = pd.concat([bos_df, msa_df], ignore_index=True)
    print(f"Loaded {len(bos_df):,} BOS rows and {len(msa_df):,} MSA rows — combined: {len(combined_df):,} rows.")

    # Filter by state
    bls_df = combined_df[combined_df['PRIM_STATE'] == TARGET_STATE].copy()
    if bls_df.empty:
        print(f"❌ No data found for state '{TARGET_STATE}'. Verify the two-letter state code.")
    else:
        print(f"Filtered to state '{TARGET_STATE}': {len(bls_df):,} rows.")

        # Optionally filter by area
        area_ready = True
        if TARGET_AREA:
            area_df = bls_df[bls_df['AREA_TITLE'] == TARGET_AREA].copy()
            if area_df.empty:
                print(f"❌ No data found for area '{TARGET_AREA}' in '{TARGET_STATE}'.")
                print("Run the discovery cell (Step 1b) to see valid area names.")
                area_ready = False
            else:
                bls_df = area_df
                print(f"Filtered to area '{TARGET_AREA}': {len(bls_df):,} rows.")
        else:
            areas_found = bls_df['AREA_TITLE'].nunique()
            print(f"No TARGET_AREA specified — retaining all {areas_found} area(s) in '{TARGET_STATE}'.")

        if area_ready:
            # Filter to cross-industry aggregate rows to prevent fan-out during the SOC code merge
            pre_filter_len = len(bls_df)
            bls_df = bls_df[bls_df['I_GROUP'] == 'cross-industry'].copy()
            print(f"Filtered to cross-industry rows: retained {len(bls_df):,} of {pre_filter_len:,} rows.")

            print("\n✅ BLS data loaded and filtered successfully!")
            display(bls_df.head(3))

except FileNotFoundError as e:
    print(f"❌ File not found: {e}")
    print(f"   Ensure {bos_file} and {msa_file} are in the same directory as this notebook.")
except Exception as e:
    print(f"❌ Error reading BLS files: {e}")

## Step 3: Organize and Handle the BLS Excel Data

The BLS spreadsheet contains many data columns, but we primarily care about the LMI statistics that provide career context:
* `OCC_CODE`: The SOC Code (our join key)
* `OCC_TITLE`: The official occupation title
* `TOT_EMP`: Total area employment for the occupation
* `JOBS_1000`: Jobs per 1,000 total employment in the area (a concentration measure)
* `LOC_QUOTIENT`: Location quotient — ratio of area concentration to the national average
* `A_MEAN`: Annual mean wage
* `A_MEDIAN`: Annual median wage
* `A_PCT25`: Annual 25th percentile wage
* `A_PCT75`: Annual 75th percentile wage
* `AREA_TITLE`: The geographic area name
* `PRIM_STATE`: The primary state for the area

We will extract these columns and handle missing values. The BLS uses special symbols (`'*'`, `'**'`, `'#'`, `'~'`) to indicate suppressed or unavailable data. Rather than dropping these values or replacing them with NaN, we preserve the original symbol meaning in a companion `_note` column (e.g., `total_employment_note`) while coercing the numeric column itself to NaN. Valid numbers are converted to floats. Any duplicate rows for the same area and SOC code are also removed so they cannot duplicate CareerNet rows during the merge.

*Notes:*
- Update the below variables to match your data of interest
    * `LMI_COLUMN_MAPPING` — BLS LMI variables of interest
    * `NUMERIC_BLS_COLS` — BLS LMI variables that should be treated as numeric
- BLS doesn't publish all minor group/broad occupation codes

In [ ]:
# ==============================================================================
# CONFIGURATION: Define your LMI Columns here
# ==============================================================================
LMI_COLUMN_MAPPING = {
    'OCC_CODE'    : 'soc_code',
    'OCC_TITLE'   : 'soc_title',
    'TOT_EMP'     : 'total_employment',
    'JOBS_1000'   : 'jobs_per_1000',
    'LOC_QUOTIENT': 'location_quotient',
    'A_MEAN'      : 'annual_mean_wage',
    'A_MEDIAN'    : 'annual_median_wage',
    'A_PCT25'     : 'annual_25percent_wage',
    'A_PCT75'     : 'annual_75percent_wage',
    'AREA_TITLE'  : 'area',
    'PRIM_STATE'  : 'state'
}

NUMERIC_BLS_COLS = ['TOT_EMP', 'JOBS_1000', 'LOC_QUOTIENT', 'A_MEAN', 'A_MEDIAN', 'A_PCT25', 'A_PCT75']
# ==============================================================================

lmi_columns = list(LMI_COLUMN_MAPPING.keys())

# Check if these columns exist to avoid KeyError
missing_cols = [col for col in lmi_columns if col not in bls_df.columns]
if missing_cols:
    print(f"Warning: The following columns are missing from the BLS data: {missing_cols}")
else:
    lmi_df = bls_df[lmi_columns].copy()

    # Define the symbol mapping based on BLS documentation
    symbol_map = {
        '*' : 'wage estimate is not available',
        '**': 'employment estimate is not available',
        '#' : 'wage equal to or greater than $115.00 per hour or $239,200 per year',
        '~' : 'the percent of establishments reporting the occupation is less than 0.5%'
    }

    # Process each numeric column: map BLS symbols to notes, coerce valid values to float
    for col in NUMERIC_BLS_COLS:
        if col in lmi_df.columns:
            lmi_df[col] = lmi_df[col].astype(str).str.strip()
            lmi_df[col + '_note'] = lmi_df[col].map(symbol_map)
            lmi_df[col] = pd.to_numeric(lmi_df[col], errors='coerce')

    # Rename columns based on the dynamic mapping dictionary
    lmi_df.rename(columns=LMI_COLUMN_MAPPING, inplace=True)

    # Rename _note columns to match their renamed base column
    note_col_rename = {
        f'{old_col}_note': f'{new_col}_note'
        for old_col, new_col in LMI_COLUMN_MAPPING.items()
        if old_col in NUMERIC_BLS_COLS
    }
    lmi_df.rename(columns=note_col_rename, inplace=True)

    # Guard against duplicate BLS rows for the same area and SOC code, which would
    # otherwise duplicate CareerNet rows during the merge
    dedup_keys = [col for col in ('area', 'soc_code') if col in lmi_df.columns]
    dup_count = lmi_df.duplicated(subset=dedup_keys).sum()
    if dup_count:
        lmi_df = lmi_df.drop_duplicates(subset=dedup_keys).copy()
        print(f"Removed {dup_count} duplicate area/SOC code rows from the BLS data.")

    print("LMI Data organized! BLS symbols replaced with string notes in their respective columns:")

    note_cols = [col + '_note' for col in NUMERIC_BLS_COLS if (col + '_note') in lmi_df.columns]
    note_mask = pd.Series(False, index=lmi_df.index)
    for col in note_cols:
        note_mask = note_mask | lmi_df[col].notna()

    rows_with_notes = lmi_df[note_mask]
    if not rows_with_notes.empty:
        display(rows_with_notes.head(3))
    else:
        display(lmi_df.head(3))

## Step 4: Load the CareerNet Dataset

Next, we load the CareerNet dataset. CareerNet contains multiple CSVs across different domains (General, Technology, Allied Health). This code points directly to the raw CSV URL on GitHub to obtain the latest CareerNet data.

*Note: Update the below variables to match your data of interest*
- `SELECTED_DATASETS` — the CareerNet domains of interest

In [ ]:
# ==============================================================================
# CONFIGURATION: Select the CareerNet datasets you want to process.
# Options available: 'general', 'health', 'technology'
# ==============================================================================
SELECTED_DATASETS = ['general', 'health', 'technology']

careernet_dfs = {}

print("=== DOWNLOADING DATASETS ===")
for ds in SELECTED_DATASETS:
    # Construct the raw GitHub URL using the user's keyword
    url = f'https://raw.githubusercontent.com/RenaissancePhilanthropy/careernet-data/main/Datasets/{ds}_public_v1.1.csv'
    try:
        print(f"Loading '{ds}' dataset from GitHub...")
        df = pd.read_csv(url)
        careernet_dfs[ds] = df
        print(f"✅ Success! ({len(df)} rows loaded)")
    except Exception as e:
        print(f"❌ Error loading '{ds}' dataset: {e}. Please check the spelling or URL.")

if careernet_dfs:
    sample_key = list(careernet_dfs.keys())[0]
    print(f"\nPreview of '{sample_key}' dataset:")
    display(careernet_dfs[sample_key].head(2))

## Step 5: Standardize SOC Codes

To merge the datasets, the SOC codes in both DataFrames must match perfectly. BLS formats SOC codes like `11-1011`. CareerNet allows for multiple SOC codes per row, separated by semicolons, and stores the SOC descriptive label alongside the code.

This step parses and explodes the CareerNet SOC codes into individual rows, validates each token against the SOC format (`##-####`), and discards any values that do not match (such as `"No career mentioned in question"`). Each SOC string lists the full hierarchy (major; minor; broad; detailed) for every detailed occupation, so parent codes such as `29-0000` can repeat many times within one row; each code is kept only once per record. Non-matching rows are retained in the output but will have no LMI data attached.

A `row_id` is also assigned to each original CareerNet row before exploding, so the validation step in Step 8 can confirm no original records were lost.

In [ ]:
careernet_exploded_dfs = {}

if 'careernet_dfs' in locals() and careernet_dfs:
    print("=== STANDARDIZING & EXPLODING SOC CODES ===")

    # Standardize BLS SOC codes once
    lmi_df['soc_code'] = lmi_df['soc_code'].astype(str).str.strip()

    # SOC code pattern: two digits, hyphen, four digits (e.g., 11-1011)
    soc_pattern = re.compile(r'^\d{2}-\d{4}$')

    def extract_soc_codes(soc_string):
        if pd.isna(soc_string) or str(soc_string).strip().lower() == 'nan':
            return [np.nan]
        parts = str(soc_string).split(';')
        codes = [part.strip().split(' ')[0] for part in parts
                 if part.strip() and soc_pattern.match(part.strip().split(' ')[0])]
        # A row's SOC string repeats parent codes once per detailed occupation;
        # keep each code once so a record does not get identical duplicate rows
        codes = list(dict.fromkeys(codes))
        return codes if codes else [np.nan]

    for ds_name, df in careernet_dfs.items():
        if 'soc_code' not in df.columns:
            print(f"❌ Error: 'soc_code' column not found in '{ds_name}'. Skipping.")
            continue

        df = df.copy()
        if 'row_id' not in df.columns:
            df['row_id'] = df.index

        df['matched_soc_code'] = df['soc_code'].apply(extract_soc_codes)
        exploded_df = df.explode('matched_soc_code')
        exploded_df['matched_soc_code'] = exploded_df['matched_soc_code'].apply(
            lambda x: str(x).strip() if pd.notna(x) else np.nan
        )

        careernet_exploded_dfs[ds_name] = exploded_df
        print(f"✅ '{ds_name}': {len(df)} original rows -> {len(exploded_df)} exploded rows.")

## Step 6: Merge LMI Data into CareerNet

Now we Join. This keeps all exploded rows from the CareerNet dataset (one row per SOC code per original record) and appends the relevant LMI statistics from the BLS dataset wherever the SOC codes match.

In [ ]:
enriched_careernet_dfs = {}

if 'careernet_exploded_dfs' in locals() and careernet_exploded_dfs:
    print("=== MERGING WITH BLS LMI DATA ===")

    for ds_name, exploded_df in careernet_exploded_dfs.items():
        enriched_df = pd.merge(
            exploded_df,
            lmi_df,
            left_on='matched_soc_code',
            right_on='soc_code',
            how='left'
        )

        # Both inputs have a soc_code column, so pandas suffixes them after merge;
        # rename to descriptive names and drop the redundant BLS copy.
        enriched_df.rename(columns={'soc_code_x': 'original_soc_string'}, inplace=True)
        enriched_df.drop(columns=['soc_code_y'], inplace=True)

        enriched_careernet_dfs[ds_name] = enriched_df
        print(f"✅ '{ds_name}' successfully merged!")

    if enriched_careernet_dfs:
        sample_key = list(enriched_careernet_dfs.keys())[0]
        print(f"\nPreview of '{sample_key}' enriched dataset:")
        display(enriched_careernet_dfs[sample_key].head(2))

## Step 7: Export the Enriched Dataset

Export the augmented dataset to a new CSV file so it can be used for modeling, analytics, or further application development.

*Note: The output filename follows the pattern `{dataset}-careernet_{area}-lmi-{year}.csv` (e.g., `general-careernet_st-louis-mo-il-lmi-2025.csv`). When no `TARGET_AREA` is set, the state code is used in place of the area slug.*

*Running in Google Colab?* The CSVs are saved to `/content`. Download them from the **Files** panel (folder icon in the left sidebar) before the runtime resets, or they will be lost.

In [ ]:
if 'enriched_careernet_dfs' in locals() and enriched_careernet_dfs:
    print("=== EXPORTING DATASETS ===")

    # Extract the 4-digit year from BLS_YEAR_SUFFIX (e.g., 'M2025_dl.xlsx' -> '2025')
    year_match = re.search(r'\d{4}', BLS_YEAR_SUFFIX)
    bls_year = year_match.group(0) if year_match else 'unknown-year'

    # Build a lowercase, hyphen-separated slug from the area (or fall back to state code)
    area_slug = (
        TARGET_AREA.lower().replace(', ', '-').replace(' ', '-').replace('.', '')
        if TARGET_AREA else TARGET_STATE.lower()
    )

    for ds_name, enriched_df in enriched_careernet_dfs.items():
        output_filename = f'{ds_name}-careernet_{area_slug}-lmi-{bls_year}.csv'
        enriched_df.to_csv(output_filename, index=False)
        print(f"💾 Successfully saved: {output_filename}")

## Step 8: Validation and Quality Assurance

Confirm the merge behaved exactly as expected. Checks that no original CareerNet records were lost and that all expected LMI columns are present in the enriched dataset.

In [ ]:
if 'enriched_careernet_dfs' in locals() and 'careernet_dfs' in locals():
    print("=== VALIDATION CHECK ===")

    # Dynamically get the list of expected new columns
    expected_lmi_cols = [new_col for old_col, new_col in LMI_COLUMN_MAPPING.items() if old_col != 'OCC_CODE']
    new_numeric_cols = [LMI_COLUMN_MAPPING[col] for col in NUMERIC_BLS_COLS if col in LMI_COLUMN_MAPPING]

    for ds_name in enriched_careernet_dfs.keys():
        print(f"\n--- Validating '{ds_name}' ---")
        original_df = careernet_dfs[ds_name]
        enriched_df = enriched_careernet_dfs[ds_name]

        # 1. Preservation Check
        original_row_count = len(original_df)
        unique_original_rows = enriched_df['row_id'].nunique()

        if unique_original_rows == original_row_count:
            print(f"✅ Rows Preserved: {original_row_count} unique records strictly accounted for.")
        else:
            print(f"❌ FAIL: Expected {original_row_count} unique rows, found {unique_original_rows}.")

        # 2. LMI Column Check
        missing_cols = [col for col in expected_lmi_cols if col not in enriched_df.columns]
        if not missing_cols:
            check_col = new_numeric_cols[0] if new_numeric_cols else expected_lmi_cols[0]
            matched_records = enriched_df[check_col].notna().sum()
            total_records = len(enriched_df)
            print(f"✅ Data Mapped: Successfully attached BLS stats to {matched_records} out of {total_records} expanded rows.")
        else:
            print(f"❌ FAIL: Missing LMI columns: {missing_cols}")